In [0]:
print("bronze:    ", spark.table("nyc_taxi.bronze.yellow_tripdata").count())      # expect 41,169,720
print("silver clean:", spark.table("nyc_taxi.silver.trips_clean").count())          # expect 35,627,370
print("silver quarantine:", spark.table("nyc_taxi.silver.trips_quarantine").count())# expect 5,542,350

In [0]:
from pyspark.sql import functions as F

silver = spark.table("nyc_taxi.silver.trips_clean")
zone_keys = [r.zone_key for r in spark.table("nyc_taxi.gold.dim_taxi_zone").select("zone_key").collect()]

fact = (silver
    .withColumn("date_key", F.col("pickup_date"))
    .withColumn("pickup_zone_key",
        F.when(F.col("PULocationID").isin(zone_keys), F.col("PULocationID")).otherwise(F.lit(-1)))
    .withColumn("dropoff_zone_key",
        F.when(F.col("DOLocationID").isin(zone_keys), F.col("DOLocationID")).otherwise(F.lit(-1)))
    .withColumn("rate_code_key",
        F.coalesce(F.col("RatecodeID").cast("int"), F.lit(-1)))
    .withColumn("payment_type_key",
        F.coalesce(F.col("payment_type").cast("int"), F.lit(-1)))
    .select(
        "trip_id", "date_key", "pickup_zone_key", "dropoff_zone_key",
        "rate_code_key", "payment_type_key",
        "tpep_pickup_datetime", "tpep_dropoff_datetime",
        "passenger_count", "trip_distance", "trip_duration_min",
        "fare_amount", "tip_amount", "total_amount")
)

(fact.write.format("delta").partitionBy("date_key").saveAsTable("nyc_taxi.gold.fact_trips"))
print("fact_trips rows:", spark.table("nyc_taxi.gold.fact_trips").count())